# Compute Routing

## What you'll learn

- Understand the difference between step runners and compute routing
- Switch a step's compute provider with the `compute_provider` parameter
- Set pipeline-wide defaults with `default_compute_provider`
- Mix compute providers in a single pipeline
- Know which operations can run remotely (command ops)

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Step Overrides](../05-errors-and-control/01-step-overrides.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [2]:
env = tutorial_setup("compute_routing")

## Step runner vs compute routing

Artisan has two orthogonal axes for controlling where work runs:

```
  Step runner                   Compute provider
  (where the worker runs)       (where the execute phase runs inside the worker)

  ┌─────────────────┐           ┌─────────────────┐
  │  Local process   │──────────│  Local call      │  (default)
  │  Provider runner │──────────│  Modal container │
  │                  │          │  (future)        │
  └─────────────────┘           └─────────────────┘
```

| Axis | Controls | Parameter | Example |
|------|----------|-----------|--------|
| Step runner | Where the *worker process* runs | `step_runner` | `Runner.LOCAL`, an optional provider instance |
| Compute provider (`compute_provider`) | Where the *execute phase* runs inside the worker | `compute_provider` | `"local"`, `"modal"` |

These axes are independent. A provider worker can route the execute phase to Modal.
A local worker can also route the execute phase to Modal. The step runner
handles scheduling and sandbox setup; compute routing handles only the
execute phase itself.

## The `compute_provider` parameter

Every `pipeline.run()` and `pipeline.submit()` call accepts a `compute_provider`
parameter. Set it to route the execute phase to a different provider. Everything
else — operation class, inputs, params, output wiring — stays identical.

```python
# Local (default)
step = pipeline.run(MyOp, inputs=..., compute_provider="local")

# Modal — same operation, same inputs, different compute provider
step = pipeline.run(MyOp, inputs=..., compute_provider="modal")
```

This means you can develop and debug locally, then move heavy compute to
Modal by changing one argument per step.

In [ ]:
pipeline = PipelineManager.create(
    name="compute_routing_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute_provider="local",
)
output = pipeline.output

print(
    f"Pipeline created with default_compute_provider='{pipeline.config.default_compute_provider}'"
)

### Generate data (local compute)

DataGenerator is fast — runs locally with `compute_provider="local"`.

In [ ]:
step0 = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
    compute_provider="local",
)
print(f"Generated {step0.succeeded_count} datasets with local compute")

### Transform data

In production, you would use `compute_provider="modal"` for a GPU-intensive step.
Here we use `"local"` to demonstrate the API without requiring credentials.
The `compute_provider` parameter follows the same override pattern as `step_runner`.

In [ ]:
step1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 2, "seed": 100},
    compute_provider="local",
)
print(f"Transformed {step1.succeeded_count} datasets with local compute")

In [ ]:
step2 = pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    compute_provider="local",
)
print(f"Computed metrics for {step2.succeeded_count} datasets with local compute")

In [ ]:
summary = pipeline.finalize()
print(
    f"Pipeline complete: {summary['total_steps']} steps, "
    f"success={summary['overall_success']}"
)
inspect_pipeline(env.delta_root)

## Pipeline-wide defaults with `default_compute_provider`

Set `default_compute_provider` at pipeline creation to apply a compute provider to
all steps by default. Individual steps can still override with the
`compute_provider` parameter.

Override precedence follows the same pattern as `step_runner`:

```
Pipeline default  →  Operation class default  →  Step override (wins)
```

In [ ]:
pipeline2 = PipelineManager.create(
    name="compute_defaults_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute_provider="local",
)
output2 = pipeline2.output

# These steps inherit default_compute_provider="local" — no explicit compute_provider needed
pipeline2.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
)
pipeline2.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output2("generate", "datasets")},
)

# This step explicitly overrides the default
pipeline2.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output2("transform", "dataset")},
    compute_provider="local",  # explicit override (would be "modal" in production)
)

pipeline2.finalize()
print(
    f"Pipeline default_compute_provider: '{pipeline2.config.default_compute_provider}'"
)
print("All steps completed — first two inherited the default, last one overrode it")

## What can run on Modal: command operations

The modal provider runs **command ops** — operations that declare a
`ToolSpec` and a `execute_command()` instead of overriding `execute_function()`. The
framework's endpoint router ships the op's `Params` + input files to the
tool's deployed endpoint, which runs the command in a container and
returns the output files.

Pure-Python operations (custom `execute_function()`) run on the local provider
only — routing one to `"modal"` fails fast with a config error.

See [Running on Modal](04-modal-execution.ipynb) for the tool-op anatomy
and deployment.

In [ ]:
from artisan.operations.examples import WaitTool

print(
    f"WaitTool (ToolSpec + execute_command): is_command_op = {WaitTool().is_command_op()}"
)
print(
    f"DataGenerator (custom execute):      is_command_op = {DataGenerator().is_command_op()}"
)
print("\nOnly command ops can route to 'modal'; pure-Python ops run locally.")

## Summary

| Concept | What it does |
|---------|-------------|
| Step runner (`step_runner`) | Controls where the *worker process* runs (local or through an optional provider) |
| Compute provider (`compute_provider`) | Controls where the *execute phase* runs inside the worker (local, Modal) |
| `default_compute_provider` | Pipeline-wide default compute provider |
| Command ops | The only ops routable to `"modal"` — `ToolSpec` + `execute_command()` |

Step runner and compute routing are orthogonal — any combination works.
Develop and debug with `compute_provider="local"`, then switch to `"modal"` for
GPU work by changing one argument per step.

## Next steps

- [Running on Modal](04-modal-execution.ipynb) — Tool endpoints: deploy, run, and debug
- [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb) — All `pipeline.run()` override parameters
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference